# LlamaIndex Practice

Welcome to your hands-on practice with LlamaIndex! In this notebook, you'll learn how to:
- Create custom tools for AI agents
- Build a functional agent using Azure OpenAI
- Integrate Model Context Protocol (MCP) tools
- Implement observability and debugging
- Complete a creative challenge to apply your knowledge

## What You'll Build

In this notebook, you'll create an intelligent **Travel Planning Assistant** that can:
- Search for flight information
- Get weather forecasts for destinations
- Convert currencies for travel budgeting
- Use Model Context Protocol (MCP) for enhanced capabilities

By the end, you'll understand the complete agent creation flow with LlamaIndex and be ready to build your own creative agents!

## Step 1: Initial Setup & Installation

First, let's install the required packages. We need:
- **llama-index-core**: Core LlamaIndex framework (agents, tools, workflows)
- **llama-index-llms-openai**: OpenAI-compatible LLM integration (works with Azure OpenAI's v1 endpoint)
- **python-dotenv**: Loads credentials from a `.env` file

In [19]:
%pip install -q llama-index-core llama-index-llms-openai python-dotenv
%pip install -q "greenlet==3.1.1"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Configure Azure OpenAI Credentials

You'll need an Azure OpenAI API key, endpoint, and model deployment. Store them in a `.env` file (in this folder or any parent folder); if a value is missing, the cell below prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [20]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## Step 3: Create Custom Tools

Tools are functions that your agent can call to perform specific tasks. Let's create travel-related tools.

**Key Concepts:**
- Each tool is a Python function with clear docstrings
- Type hints help the LLM understand parameter types
- Docstrings describe what the tool does (the LLM reads this!)

In [21]:
from datetime import datetime, timedelta
import random


def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights between two cities on a specific date.

    Args:
        origin: The departure city (e.g., 'New York')
        destination: The arrival city (e.g., 'Paris')
        date: The travel date in YYYY-MM-DD format

    Returns:
        A string with flight information including price and duration
    """
    # Simulate flight search
    flight_number = f"FL{random.randint(100, 999)}"
    price = random.randint(300, 1500)
    duration = random.randint(4, 15)

    return f"Found flight {flight_number} from {origin} to {destination} on {date}. Price: ${price}, Duration: {duration}h"


def get_weather(city: str, days_ahead: int = 0) -> str:
    """Get weather forecast for a city.

    Args:
        city: The city name to get weather for
        days_ahead: Number of days ahead to forecast (0 = today, 1 = tomorrow, etc.)

    Returns:
        A string describing the weather conditions
    """
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Stormy"]
    condition = random.choice(conditions)
    temp = random.randint(15, 30)
    forecast_date = datetime.now() + timedelta(days=days_ahead)

    return f"Weather in {city} on {forecast_date.strftime('%Y-%m-%d')}: {condition}, {temp}°C"


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert
        from_currency: Source currency code (e.g., 'USD')
        to_currency: Target currency code (e.g., 'EUR')

    Returns:
        A string with the converted amount
    """
    # Simplified exchange rates (in practice, you'd use a real API)
    rates = {
        ("USD", "EUR"): 0.92,
        ("USD", "GBP"): 0.79,
        ("USD", "JPY"): 149.50,
        ("EUR", "USD"): 1.09,
        ("GBP", "USD"): 1.27,
    }

    rate = rates.get((from_currency, to_currency), 1.0)
    converted = amount * rate

    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"

## Step 4: Initialize the Language Model

Now let's set up Azure OpenAI as our LLM. Azure OpenAI's v1 API is OpenAI-compatible, so we use LlamaIndex's `OpenAI` class with `api_base` pointing at the Azure endpoint.

**Note:** `model` is your deployment name. It must match an OpenAI model name (e.g. `gpt-5.4-mini`, `gpt-5.6-luna`) so LlamaIndex knows the model's context window and tool-calling support.

In [22]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(
    model=AZURE_OPENAI_DEPLOYMENT,
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

## Step 5: Create Your First Agent

A **FunctionAgent** is LlamaIndex's agentic workflow that can:
1. Understand user queries
2. Decide which tools to use
3. Call tools with appropriate parameters
4. Synthesize responses from tool outputs

Let's create our Travel Planning Agent!

In [23]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm,
    verbose=True  # This helps us see what the agent is doing
)

## Step 6: Create an Observability Helper

Before testing our agent, let's create a helper function that gives us **detailed visibility** into what the agent is doing.

**Why Observability Matters:**
- 🔍 See which tools the agent chooses
- 📊 Inspect the inputs passed to each tool
- ✅ Verify the outputs returned
- 🐛 Debug issues when the agent doesn't behave as expected

The `run_agent_verbose()` function will:
1. Display the user's query
2. Stream events as the agent works
3. Print details about each tool call
4. Return the final response

This is crucial for understanding and debugging agent behavior!

In [24]:
from llama_index.core.agent.workflow import ToolCallResult


async def run_agent_verbose(query: str):
    """Run the agent and print detailed information about tool calls."""
    print(f"🤖 User Query: {query}\n")
    print("=" * 60)

    handler = agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 Tool Used: {event.tool_name}")
            print(f"📥 Input: {event.tool_kwargs}")
            print(f"📤 Output: {event.tool_output}")
            print("-" * 60)

    result = await handler
    print(f"\n✨ Final Response:\n{result}\n")
    return result

## Step 7: Test Your Agent with Simple Queries

Let's start with basic queries to see how the agent uses tools:

In [25]:
response = await run_agent_verbose(
    "What's the weather like in Tokyo today?"
)

🤖 User Query: What's the weather like in Tokyo today?

[tick] add: AgentWorkflowStartEvent(user_msg="What's the weather like in Tokyo today?", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, addit

In [26]:
# Inspect which tools were called
print("Tool Calls Made:")
for tool_call in response.tool_calls:
    print(f"  - {tool_call}")

Tool Calls Made:
  - tool_name='get_weather' tool_kwargs={'city': 'Tokyo', 'days_ahead': 0} tool_id='call_gYLNpUVFhfP3znU6Eif4CScO' tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Weather in Tokyo on 2026-11-23: Sunny, 23°C')], tool_name='get_weather', raw_input={'args': (), 'kwargs': {'city': 'Tokyo', 'days_ahead': 0}}, raw_output='Weather in Tokyo on 2026-11-23: Sunny, 23°C', is_error=False) return_direct=False


## Step 8: Complex Multi-Tool Query

Watch how the agent chains multiple tools together to answer complex questions:

In [27]:
response = await run_agent_verbose(
    "I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?"
)

🤖 User Query: I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?

[tick] add: AgentWorkflowStartEvent(user_msg='I want to fly from London to Barcelona on 2026-12-...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] 

## Step 9: Managing Context & Memory

By default, `.run()` is stateless (doesn't remember previous conversations). Let's add memory using a **Context** object:

In [28]:
from llama_index.core.workflow import Context

# Create a new agent instance for this demo
memory_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm
)

# Create a context to maintain conversation history
ctx = Context(memory_agent)

# First message - introduce yourself
response1 = await memory_agent.run(
    "Hi! I'm planning a trip to Rome in December. My name is Alex.",
    ctx=ctx
)
print(f"Response 1: {response1}\n")

# Second message - agent should remember your name
response2 = await memory_agent.run(
    "What's my name, and what's the weather like in my destination?",
    ctx=ctx
)
print(f"Response 2: {response2}")

Response 1: Hi Alex! Rome in December can be lovely.

If you’d like, I can help you plan things like:
- flight options
- weather for your travel dates
- what to pack
- itinerary ideas
- budget estimates

If you want, tell me:
1. your departure city,
2. your exact dates, and
3. whether you want a budget, mid-range, or comfort trip.

Response 2: I don’t know your name yet. If you tell me, I can use it.

I also need to know your destination city to check the weather there.


## Step 10: Advanced - Model Context Protocol (MCP) Tools

MCP is a standardized protocol for connecting AI models to external tools and data sources. Let's integrate an MCP tool!

**What is MCP?**
- Open protocol for AI-to-tool communication
- Enables standardized tool integration
- Supports local and remote tools

### Creating a Custom MCP-Style Tool

For this practice, we simulate an MCP tool that fetches travel tips with a regular `FunctionTool`.

To load tools from a real MCP server, install `llama-index-tools-mcp` and convert the server's tools into LlamaIndex tools:
```python
# from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
# mcp_client = BasicMCPClient("http://127.0.0.1:8000/mcp")  # or BasicMCPClient("python", args=["server.py"]) for stdio
# mcp_tools = await McpToolSpec(client=mcp_client).to_tool_list_async()
```

In [29]:
from llama_index.core.tools import FunctionTool


def get_travel_tips(destination: str) -> str:
    """Get essential travel tips for a destination.

    Args:
        destination: The city or country to get tips for

    Returns:
        A string with helpful travel tips
    """
    tips_database = {
        "paris": "🇫🇷 Tips for Paris: Try croissants at local boulangeries, use the Metro, visit Eiffel Tower early morning",
        "tokyo": "🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa",
        "new york": "🇺🇸 Tips for NYC: Use subway, try pizza and bagels, visit Central Park and Times Square",
        "barcelona": "🇪🇸 Tips for Barcelona: Visit Sagrada Familia, try tapas and paella, walk Las Ramblas",
        "rome": "🇮🇹 Tips for Rome: Visit Colosseum early, try authentic carbonara, throw coin in Trevi Fountain",
    }

    city_lower = destination.lower()
    for key in tips_database:
        if key in city_lower:
            return tips_database[key]

    return f"💡 General tips: Research local customs, learn basic phrases, try local cuisine!"


# Convert to LlamaIndex tool
travel_tips_tool = FunctionTool.from_defaults(fn=get_travel_tips)

# Create agent with MCP-style tool
mcp_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency, travel_tips_tool],
    llm=llm
)

print("✅ Agent with MCP tool created!")

✅ Agent with MCP tool created!


### Testing the Enhanced Agent

In [30]:
async def run_mcp_agent(query: str):
    """Run the MCP-enhanced agent."""
    print(f"🤖 Query: {query}\n")
    handler = mcp_agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 {event.tool_name} → {event.tool_output}")

    result = await handler
    print(f"\n✨ {result}\n")
    return result


response = await run_mcp_agent(
    "I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow."
)

🤖 Query: I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow.

🔧 get_weather → Weather in Tokyo on 2026-11-24: Stormy, 19°C
🔧 get_travel_tips → 🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa

✨ Here you go:

**Tomorrow’s weather in Tokyo:** Stormy, **19°C**

**Travel tips for Tokyo:**
- Get a **Suica card** for easy transport
- Try **ramen** at local shops
- Visit the **temples in Asakusa**

If you want, I can also help with packing suggestions based on the weather.



## Step 11: Enhanced Observability with Custom Callbacks

Let's implement more detailed observability to track agent performance:

In [31]:
import time
from typing import List


class AgentObserver:
    """Track and display agent performance metrics."""

    def __init__(self):
        self.tool_calls = []
        self.start_time = None
        self.end_time = None

    async def observe_agent(self, query: str, agent: FunctionAgent):
        """Run agent and collect observability data."""
        self.tool_calls = []
        self.start_time = time.time()

        print(f"🎯 Query: {query}\n")
        print("📊 Observability Dashboard")
        print("=" * 60)

        handler = agent.run(query)
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                self.tool_calls.append({
                    'tool': event.tool_name,
                    'input': event.tool_kwargs,
                    'output': event.tool_output
                })
                print(f"⚙️  Tool Call #{len(self.tool_calls)}: {event.tool_name}")

        result = await handler
        self.end_time = time.time()

        # Display metrics
        print("\n" + "=" * 60)
        print("📈 Performance Metrics:")
        print(f"  ⏱️  Total Time: {self.end_time - self.start_time:.2f}s")
        print(f"  🔧 Tools Called: {len(self.tool_calls)}")
        print(f"  📝 Tool Names: {[t['tool'] for t in self.tool_calls]}")
        print(f"\n💬 Response: {result}")

        return result


# Test the observer
observer = AgentObserver()
await observer.observe_agent(
    "Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY",
    mcp_agent
)

🎯 Query: Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY

📊 Observability Dashboard
⚙️  Tool Call #1: search_flights
⚙️  Tool Call #2: get_weather
⚙️  Tool Call #3: convert_currency

📈 Performance Metrics:
  ⏱️  Total Time: 2.93s
  🔧 Tools Called: 3
  📝 Tool Names: ['search_flights', 'get_weather', 'convert_currency']

💬 Response: Here’s what I found:

- Flight: Paris → Tokyo on 2026-12-20
  - Flight FL971
  - Price: $460
  - Duration: 7h

- Weather in Tokyo:
  - Rainy, 17°C

- Currency conversion:
  - 1000 EUR = 1000.00 JPY

If you want, I can also help compare more flight options or give travel tips for Tokyo.


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Here’s what I found:\n\n- Flight: Paris → Tokyo on 2026-12-20\n  - Flight FL971\n  - Price: $460\n  - Duration: 7h\n\n- Weather in Tokyo:\n  - Rainy, 17°C\n\n- Currency conversion:\n  - 1000 EUR = 1000.00 JPY\n\nIf you want, I can also help compare more flight options or give travel tips for Tokyo.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-EnOkvIz9316Cf7hEvs3EU18U6NzUz', 'choices': [{'delta': {'audio': None, 'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'index': 0, 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {}}], 'created': 1795441941, 'model': 'gpt-5.4-mini-2026-03-17', 'object': 'chat.completion.chunk', 'moderation': None, 'obfuscation': 'K7bkJUs8eWvE', 'service_tier': 'default', 'system_fingerprint': None, 'usage': None}, tool_calls=[To

## 🎉 Congratulations!

You've learned how to:
- ✅ Create custom tools for AI agents
- ✅ Build functional agents with LlamaIndex
- ✅ Implement observability and debugging
- ✅ Manage conversation context and memory
- ✅ Integrate MCP-style tools
- ✅ Track agent performance metrics

### Share Your Work:
- Complete the challenge and share your restaurant agent
- Experiment with different tools and domains
- Try integrating real APIs for production use

Happy building! 🚀

## 🎓 Challenge: Build Your Own Agent!

Now it's your turn! Apply what you've learned to create a unique agent.

### Your Mission:
Create a **Restaurant Recommendation Agent** that helps users find and learn about restaurants.

### Requirements:
1. **Create at least 3 custom tools:**
   - `find_restaurants(cuisine: str, city: str) -> str`: Find restaurants by cuisine type
   - `get_restaurant_details(restaurant_name: str) -> str`: Get details like hours, price range
   - `make_reservation(restaurant_name: str, date: str, party_size: int) -> str`: Simulate booking

2. **Add a bonus tool** (get creative!):
   - Menu translator
   - Dietary restriction filter
   - Restaurant reviews summarizer
   - Distance calculator
   - Or your own idea!

3. **Implement:**
   - Agent with all your tools
   - Observability (print tool calls)
   - Context/memory for conversation
   - Test with at least 2 complex queries

4. **Bonus Points:**
   - Make it work with real APIs (like Yelp or Google Places)
   - Add error handling
   - Create a user-friendly interface
   - Implement caching for repeated queries

### Tips:
- Use clear docstrings - the LLM reads them!
- Test each tool individually first
- Start simple, then add complexity
- Use type hints for better tool understanding
- Think about real-world use cases

### Example Test Queries:
- "Find Italian restaurants in Boston and get details about the top one"
- "I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people tomorrow"
- "Recommend a restaurant with outdoor seating, check if they're open now, and book a table"

### Your Solution Space

Write your code below:

In [32]:
# TODO: Define your restaurant tools here

# Mock restaurant database (in practice, you'd use a real API like Yelp)
RESTAURANTS = {
    "Trattoria Roma": {"cuisine": "italian", "city": "boston", "hours": "12:00 to 23:00",
                       "price": "$$", "vegetarian": True, "outdoor": True},
    "Bella Napoli": {"cuisine": "italian", "city": "boston", "hours": "17:00 to 22:00",
                     "price": "$$$", "vegetarian": False, "outdoor": False},
    "Green Leaf": {"cuisine": "vegetarian", "city": "seattle", "hours": "11:00 to 21:00",
                   "price": "$", "vegetarian": True, "outdoor": True},
    "Sakura Sushi": {"cuisine": "japanese", "city": "seattle", "hours": "12:00 to 22:00",
                     "price": "$$", "vegetarian": False, "outdoor": False},
}


def find_restaurants(cuisine: str, city: str) -> str:
    """Find restaurants by cuisine type in a city.

    Args:
        cuisine: The type of food (e.g., 'italian', 'japanese', 'vegetarian')
        city: The city to search in (e.g., 'Boston')

    Returns:
        A string with the names of matching restaurants
    """
    found = [name for name, info in RESTAURANTS.items()
             if info["cuisine"] == cuisine.lower() and info["city"] == city.lower()]
    if not found:
        return f"No {cuisine} restaurants found in {city}."
    return f"{cuisine.title()} restaurants in {city}: {', '.join(found)}"


def get_restaurant_details(restaurant_name: str) -> str:
    """Get details about a restaurant like opening hours and price range.

    Args:
        restaurant_name: The exact name of the restaurant

    Returns:
        A string with the restaurant details
    """
    info = RESTAURANTS.get(restaurant_name)
    if info is None:
        return f"Error: restaurant '{restaurant_name}' not found."
    return (f"{restaurant_name}: hours {info['hours']}, price range {info['price']}, "
            f"outdoor seating: {'yes' if info['outdoor'] else 'no'}")


def make_reservation(restaurant_name: str, date: str, party_size: int) -> str:
    """Make a table reservation at a restaurant (simulated).

    Args:
        restaurant_name: The exact name of the restaurant
        date: The reservation date in YYYY-MM-DD format
        party_size: Number of people

    Returns:
        A confirmation message with a booking number
    """
    if restaurant_name not in RESTAURANTS:
        return f"Error: restaurant '{restaurant_name}' not found."
    if party_size < 1:
        return "Error: party size must be at least 1."
    booking_id = f"RES{random.randint(1000, 9999)}"
    return f"Reservation confirmed at {restaurant_name} on {date} for {party_size} people. Booking ID: {booking_id}"


# TODO: Add your bonus tool
def check_dietary_options(restaurant_name: str, diet: str) -> str:
    """Check if a restaurant fits a dietary restriction.

    Args:
        restaurant_name: The exact name of the restaurant
        diet: The dietary restriction (currently supports 'vegetarian')

    Returns:
        A string saying if the restaurant fits the diet
    """
    info = RESTAURANTS.get(restaurant_name)
    if info is None:
        return f"Error: restaurant '{restaurant_name}' not found."
    if diet.lower() == "vegetarian":
        return f"{restaurant_name} {'has' if info['vegetarian'] else 'does NOT have'} vegetarian options."
    return f"No information about '{diet}' options for {restaurant_name}."


# Test each tool individually first
print(find_restaurants("italian", "Boston"))
print(get_restaurant_details("Trattoria Roma"))
print(make_reservation("Green Leaf", "2026-11-24", 4))
print(check_dietary_options("Bella Napoli", "vegetarian"))

Italian restaurants in Boston: Trattoria Roma, Bella Napoli
Trattoria Roma: hours 12:00 to 23:00, price range $$, outdoor seating: yes
Reservation confirmed at Green Leaf on 2026-11-24 for 4 people. Booking ID: RES7837
Bella Napoli does NOT have vegetarian options.


In [33]:
# TODO: Create your agent
restaurant_agent = FunctionAgent(
    tools=[find_restaurants, get_restaurant_details, make_reservation, check_dietary_options],
    llm=llm,
    system_prompt="You are a helpful restaurant assistant. Always use the tools to get restaurant information.",
)

# Context keeps the conversation history (memory)
restaurant_ctx = Context(restaurant_agent)

# Explicit memory object: stores the chat history between runs
from llama_index.core.memory import Memory
restaurant_memory = Memory.from_defaults(session_id="restaurant_chat", token_limit=40000)


async def run_restaurant_agent(query: str):
    """Run the agent with memory and print every tool call (observability)."""
    print(f"🤖 User Query: {query}\n")
    print("=" * 60)

    handler = restaurant_agent.run(query, ctx=restaurant_ctx, memory=restaurant_memory)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 Tool Used: {event.tool_name}")
            print(f"📥 Input: {event.tool_kwargs}")
            print(f"📤 Output: {event.tool_output}")
            print("-" * 60)

    result = await handler
    print(f"\n✨ Final Response:\n{result}\n")
    return result

In [34]:
# TODO: Test your agent with complex queries

# Query 1: find + details
await run_restaurant_agent(
    "Find Italian restaurants in Boston and get details about the top one."
)

# Query 2: diet filter + reservation
await run_restaurant_agent(
    "I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people on 2026-11-24."
)

# Query 3: memory test 
await run_restaurant_agent(
    "What is the booking ID of my reservation, and what are that restaurant's opening hours?"
)

🤖 User Query: Find Italian restaurants in Boston and get details about the top one.

🔧 Tool Used: find_restaurants
📥 Input: {'cuisine': 'italian', 'city': 'Boston'}
📤 Output: Italian restaurants in Boston: Trattoria Roma, Bella Napoli
------------------------------------------------------------
🔧 Tool Used: get_restaurant_details
📥 Input: {'restaurant_name': 'Trattoria Roma'}
📤 Output: Trattoria Roma: hours 12:00 to 23:00, price range $$, outdoor seating: yes
------------------------------------------------------------

✨ Final Response:
Here are Italian restaurants in Boston:
- Trattoria Roma
- Bella Napoli

Top result details for Trattoria Roma:
- Hours: 12:00 to 23:00
- Price range: $$
- Outdoor seating: yes

🤖 User Query: I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people on 2026-11-24.

🔧 Tool Used: find_restaurants
📥 Input: {'cuisine': 'vegetarian', 'city': 'Seattle'}
📤 Output: Vegetarian restaurants in Seattle: Green Leaf
----------------------

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Your booking ID is **RES5129**.\n\nGreen Leaf’s opening hours are **11:00 to 21:00**.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-EnOl5ST6hpvr0f0QkuXrXlG1OV7Fx', 'choices': [{'delta': {'audio': None, 'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'index': 0, 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {}}], 'created': 1795441951, 'model': 'gpt-5.4-mini-2026-03-17', 'object': 'chat.completion.chunk', 'moderation': None, 'obfuscation': '0lRygpY8LULU', 'service_tier': 'default', 'system_fingerprint': None, 'usage': None}, tool_calls=[ToolCallResult(tool_name='get_restaurant_details', tool_kwargs={'restaurant_name': 'Green Leaf'}, tool_id='call_dQFjEB8KKp6dxneenMurzW6z', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Green Leaf: